# 🚀 Titan-Optimal Training on Google Colab

This notebook trains the Titan-Optimal GPT model on GPU and saves checkpoints to Google Drive.

## Features:
- ✅ GPU acceleration (T4/V100/A100)
- ✅ Automatic Google Drive mounting
- ✅ Checkpoint saving to Drive
- ✅ Progress monitoring
- ✅ Result visualization

## Setup Requirements:
1. Enable GPU: Runtime → Change runtime type → GPU
2. Run all cells in order
3. Authorize Google Drive access when prompted

## 📋 Step 1: Check GPU Availability

In [ ]:
import torch
import sys

print("="*80)
print("GPU AVAILABILITY CHECK")
print("="*80)

if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = torch.device("cuda")
else:
    print("⚠️ GPU not available, using CPU (training will be slow!)")
    device = torch.device("cpu")

print(f"\nDevice set to: {device}")
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print("="*80)

## 📁 Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

print("="*80)
print("MOUNTING GOOGLE DRIVE")
print("="*80)

# Mount Google Drive
drive.mount('/content/drive')

# Create checkpoint directory
checkpoint_dir = "/content/drive/MyDrive/Titan_Optimal_Checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"\n✅ Google Drive mounted successfully")
print(f"✅ Checkpoint directory created: {checkpoint_dir}")
print("="*80)

## 📦 Step 3: Install Dependencies

In [ ]:
print("="*80)
print("INSTALLING DEPENDENCIES")
print("="*80)

# Install required packages
!pip install -q tiktoken matplotlib tqdm

print("\n✅ All dependencies installed")
print("="*80)

## 🌐 Step 4: Clone Repository (Colabnotebook Branch)

In [ ]:
import os

print("="*80)
print("CLONING REPOSITORY")
print("="*80)

repo_path = "/content/LLMs-from-scratch"

# Remove existing repo if present
if os.path.exists(repo_path):
    print("📁 Removing existing repository...")
    !rm -rf {repo_path}

# Clone the repository with the Colabnotebook branch
print("📥 Cloning repository (Colabnotebook branch)...")
!git clone --depth 1 --branch Colabnotebook https://github.com/YOUR_USERNAME/LLMs-from-scratch.git {repo_path}

# Verify the titan_optimal folder exists
titan_optimal_path = f"{repo_path}/titan_optimal"
if os.path.exists(titan_optimal_path):
    print(f"\n✅ Repository cloned successfully")
    print(f"✅ Titan-Optimal folder found: {titan_optimal_path}")
    
    # List files
    print("\n📂 Contents of titan_optimal/:")
    !ls -la {titan_optimal_path}
else:
    print(f"\n❌ ERROR: titan_optimal folder not found!")
    print(f"   Expected path: {titan_optimal_path}")

# Add to Python path
import sys
sys.path.append(repo_path)
sys.path.append(f"{repo_path}/ch04/01_main-chapter-code")
sys.path.append(titan_optimal_path)

print("\n✅ Paths added to system")
print("="*80)

## ⚙️ Step 5: Configuration

In [ ]:
# Training Configuration
CONFIG = {
    "model_size": "small",  # "small" (124M) or "medium" (340M)
    "num_epochs": 10,
    "batch_size": 8,  # Adjust based on GPU memory (T4: 8, V100: 12-16, A100: 16-24)
    "learning_rate": 5e-4,
    "weight_decay": 0.1,
    "gradient_accumulation_steps": 4,
    "save_every_n_epochs": 2,
    "context_length": 512,  # Small: 512, Medium: 1024
    "eval_freq": 50,
    "eval_iter": 10,
    "checkpoint_dir": "/content/drive/MyDrive/Titan_Optimal_Checkpoints"
}

print("="*80)
print("TRAINING CONFIGURATION")
print("="*80)
for key, value in CONFIG.items():
    print(f"{key:30s}: {value}")
print("="*80)

# Adjust batch size for GPU type
if device.type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    if "T4" in gpu_name:
        print("\n💡 Detected T4 GPU - using batch_size=8")
        CONFIG["batch_size"] = 8
    elif "V100" in gpu_name:
        print("\n💡 Detected V100 GPU - using batch_size=12")
        CONFIG["batch_size"] = 12
    elif "A100" in gpu_name:
        print("\n💡 Detected A100 GPU - using batch_size=16")
        CONFIG["batch_size"] = 16
else:
    print("\n⚠️ CPU detected - reducing batch size to 2")
    CONFIG["batch_size"] = 2
    CONFIG["gradient_accumulation_steps"] = 8

## 📚 Step 6: Load Training Data

In [ ]:
import requests
import os

print("="*80)
print("LOADING TRAINING DATA")
print("="*80)

data_path = "/content/LLMs-from-scratch/ch05/01_main_chapter_code/the-verdict.txt"

if not os.path.exists(data_path):
    print("📥 Downloading training data...")
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    response = requests.get(url, timeout=30)
    os.makedirs(os.path.dirname(data_path), exist_ok=True)
    with open(data_path, "w", encoding="utf-8") as f:
        f.write(response.text)
    print("✅ Training data downloaded")
else:
    print("✅ Training data already exists")

# Load text data
with open(data_path, "r", encoding="utf-8") as f:
    text_data = f.read()

print(f"\n✅ Loaded {len(text_data):,} characters")
print(f"   First 100 characters: {text_data[:100]}...")
print("="*80)

## 🔧 Step 7: Create Dataloaders

In [ ]:
import tiktoken
import sys
sys.path.append("/content/LLMs-from-scratch/ch04/01_main-chapter-code")

from gpt import create_dataloader_v1

print("="*80)
print("CREATING DATALOADERS")
print("="*80)

# Split data
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_text = text_data[:split_idx]
val_text = text_data[split_idx:]

print(f"Train text: {len(train_text):,} characters")
print(f"Val text: {len(val_text):,} characters")

# Create dataloaders
train_loader = create_dataloader_v1(
    train_text,
    batch_size=CONFIG["batch_size"],
    max_length=CONFIG["context_length"],
    stride=CONFIG["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_text,
    batch_size=CONFIG["batch_size"],
    max_length=CONFIG["context_length"],
    stride=CONFIG["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

print(f"\n✅ Train batches: {len(train_loader)}")
print(f"✅ Val batches: {len(val_loader)}")
print(f"\nBatch shape: {next(iter(train_loader))[0].shape}")
print("="*80)

## 🤖 Step 8: Create Model

In [ ]:
from models.titan_gpt_v3 import TitanGPTModelV3
from configs.model_configs_v3 import get_model_config_v3

print("="*80)
print("CREATING MODEL")
print("="*80)

# Get model config
v3_cfg = get_model_config_v3(CONFIG["model_size"], use_memory=True)
v3_cfg["batch_size"] = CONFIG["batch_size"]
v3_cfg["context_length"] = CONFIG["context_length"]

print(f"\nModel: Titan-Optimal V3 {CONFIG['model_size'].upper()}")
print(f"Configuration:")
for key in ["emb_dim", "n_heads", "n_layers", "short_term_size", "medium_term_size", "long_term_size"]:
    if key in v3_cfg:
        print(f"  {key:20s}: {v3_cfg[key]}")

# Create model
model = TitanGPTModelV3(v3_cfg).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✅ Model created successfully")
print(f"   Total parameters: {total_params:,}")
print(f"   Device: {device}")
print("="*80)

## 🎯 Step 9: Training

In [ ]:
import torch.nn as nn
import time
from tqdm import tqdm

print("="*80)
print("STARTING TRAINING")
print("="*80)

# Create optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

# Training metrics
train_losses = []
val_losses = []
perplexities = []
best_val_loss = float('inf')

model.train()
global_step = 0

for epoch in range(CONFIG["num_epochs"]):
    print(f"\n{'='*80}")
    print(f"EPOCH {epoch+1}/{CONFIG['num_epochs']}")
    print(f"{'='*80}")
    
    epoch_start = time.time()
    epoch_loss = 0.0
    
    # Training with progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch_idx, (input_batch, target_batch) in enumerate(pbar):
        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device)
        
        # Forward pass
        logits, aux_losses = model(
            input_batch,
            targets=target_batch,
            update_memory=True,
            mode="train"
        )
        
        # Loss
        loss = nn.functional.cross_entropy(
            logits.flatten(0, 1),
            target_batch.flatten()
        )
        
        # Backward
        loss = loss / CONFIG["gradient_accumulation_steps"]
        loss.backward()
        
        epoch_loss += loss.item()
        
        # Update weights
        if (batch_idx + 1) % CONFIG["gradient_accumulation_steps"] == 0:
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        # Periodic evaluation
        if global_step % CONFIG["eval_freq"] == 0:
            model.eval()
            
            # Validation loss
            val_loss = 0.0
            with torch.no_grad():
                for i, (inp, tgt) in enumerate(val_loader):
                    if i >= CONFIG["eval_iter"]:
                        break
                    inp, tgt = inp.to(device), tgt.to(device)
                    logits, _ = model(inp, update_memory=False, mode="inference")
                    val_loss += nn.functional.cross_entropy(
                        logits.flatten(0, 1), tgt.flatten()
                    ).item()
            
            val_loss /= min(CONFIG["eval_iter"], len(val_loader))
            perplexity = torch.exp(torch.tensor(val_loss)).item()
            
            val_losses.append(val_loss)
            perplexities.append(perplexity)
            
            print(f"\n  Step {global_step} | Val Loss: {val_loss:.4f} | Perplexity: {perplexity:.2f}")
            
            model.train()
    
    # Epoch summary
    epoch_time = time.time() - epoch_start
    avg_epoch_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_epoch_loss)
    
    print(f"\n✅ Epoch {epoch+1} Complete:")
    print(f"   Train Loss: {avg_epoch_loss:.4f}")
    print(f"   Val Loss: {val_losses[-1]:.4f}")
    print(f"   Perplexity: {perplexities[-1]:.2f}")
    print(f"   Time: {epoch_time:.2f}s")
    
    # Save checkpoint
    if (epoch + 1) % CONFIG["save_every_n_epochs"] == 0 or val_losses[-1] < best_val_loss:
        checkpoint_path = f"{CONFIG['checkpoint_dir']}/titan_optimal_{CONFIG['model_size']}_epoch_{epoch+1}.pth"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_epoch_loss,
            'val_loss': val_losses[-1],
            'perplexity': perplexities[-1],
        }, checkpoint_path)
        print(f"   💾 Checkpoint saved: {checkpoint_path}")
        
        if val_losses[-1] < best_val_loss:
            best_val_loss = val_losses[-1]
            best_path = f"{CONFIG['checkpoint_dir']}/titan_optimal_{CONFIG['model_size']}_best.pth"
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_epoch_loss,
                'val_loss': val_losses[-1],
                'perplexity': perplexities[-1],
            }, best_path)
            print(f"   ⭐ Best model saved: {best_path}")
    
    # Reset memory
    if hasattr(model, 'reset_memory'):
        model.reset_memory(level="short")

print("\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80)

## 📊 Step 10: Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import json

print("="*80)
print("VISUALIZING RESULTS")
print("="*80)

# Create plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training Loss
axes[0].plot(range(1, len(train_losses)+1), train_losses, marker='o', color='blue', label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation Loss
axes[1].plot(range(1, len(val_losses)+1), val_losses, marker='o', color='orange', label='Val Loss')
axes[1].set_xlabel('Evaluation Step')
axes[1].set_ylabel('Loss')
axes[1].set_title('Validation Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Perplexity
axes[2].plot(range(1, len(perplexities)+1), perplexities, marker='o', color='green', label='Perplexity')
axes[2].set_xlabel('Evaluation Step')
axes[2].set_ylabel('Perplexity')
axes[2].set_title('Perplexity (Lower is Better)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot
plot_path = f"{CONFIG['checkpoint_dir']}/training_progress.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"\n✅ Plot saved to: {plot_path}")
plt.show()

# Save results JSON
results_summary = {
    "model_size": CONFIG["model_size"],
    "total_params": total_params,
    "num_epochs": CONFIG["num_epochs"],
    "batch_size": CONFIG["batch_size"],
    "final_train_loss": train_losses[-1] if train_losses else None,
    "final_val_loss": val_losses[-1] if val_losses else None,
    "final_perplexity": perplexities[-1] if perplexities else None,
    "best_val_loss": best_val_loss,
    "train_losses": train_losses,
    "val_losses": val_losses,
    "perplexities": perplexities,
}

results_path = f"{CONFIG['checkpoint_dir']}/training_results_{CONFIG['model_size']}.json"
with open(results_path, "w") as f:
    json.dump(results_summary, f, indent=2)

print(f"✅ Results saved to: {results_path}")

# Print summary
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"Model: Titan-Optimal V3 {CONFIG['model_size'].upper()}")
print(f"Parameters: {total_params:,}")
print(f"Best Val Loss: {best_val_loss:.4f}")
print(f"Final Perplexity: {perplexities[-1]:.2f}")
print(f"\n📁 All files saved to: {CONFIG['checkpoint_dir']}")
print("   Access via: Google Drive → MyDrive → Titan_Optimal_Checkpoints")
print("="*80)

## 🎉 Training Complete!

Your model has been trained and saved to Google Drive.

### Files Saved:
- `titan_optimal_{size}_best.pth` - Best model checkpoint
- `titan_optimal_{size}_epoch_N.pth` - Periodic checkpoints
- `training_results_{size}.json` - Training metrics
- `training_progress.png` - Visualization

### Next Steps:
1. Download checkpoints from Google Drive
2. Use the model for inference
3. Fine-tune on your own data
4. Evaluate on benchmarks